# UC Dashboard Construction — Colab Analysis
Question (per rubric: time window + population + metric):
**For Fall 2025 freshman applicants from California public high schools, how does the
admission-rate penalty for applying to Computer Science (versus that campus's overall
admit rate) vary across the nine UC campuses, and which campuses punish CS applicants
the hardest?**

- Time window: Fall 2025
- Population: California public high school freshman applicants
- Metric: CS-vs-overall admit-rate penalty (percentage points) per campus

Run this in Google Colab. Upload the 5 CSVs from the event Google Drive first.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Upload the CSVs when prompted (or mount Drive). Files needed:
#   bay_area_modeling_table.csv, dashboard_data.csv,
#   uc_admissions_summary_by_ethnicity.csv, uc_freshman_admission_by_discipline.csv,
#   uc_transfer_admission_by_major.csv
from google.colab import files
uploaded = files.upload()

In [ ]:
disc = pd.read_csv('uc_freshman_admission_by_discipline.csv')
dash = pd.read_csv('dashboard_data.csv')

# Keep Fall 2025 only
d25 = disc[disc.fall_term == 2025].copy()

# Overall admit rate per campus (All disciplines row)
overall = d25[d25.broad_discipline == 'All disciplines'].set_index('campus')['admit_rate']

# CS admit rate per campus
cs = d25[d25.broad_discipline == 'Computer Science'].set_index('campus')['admit_rate']

# Penalty = CS rate - overall rate (negative = CS is harder)
penalty = (cs - overall) * 100  # percentage points

# Align on campuses present in BOTH (Merced has no CS row in source)
both = overall.index.intersection(cs.index)
penalty = (cs[both] - overall[both]) * 100
result = pd.DataFrame({
    'campus': both,
    'overall_rate': overall[both].values,
    'cs_rate': cs[both].values,
    'cs_penalty_pp': penalty.values,
}).sort_values('cs_penalty_pp')
result['cs_penalty_pp'] = result['cs_penalty_pp'].round(1)
print(result.to_string(index=False))

In [ ]:
# FINDING (concise + justifiable):
# At almost every UC campus, applying to CS lowers your admit rate vs the campus average.
# The penalty is steepest at UC Davis (-25.0 pp) and shallowest at UC Irvine (-1.0 pp).
# Santa Cruz is the exception: CS is EASIER there (79% vs 72% overall).
# Merced reports no CS row in the source, so 8 campuses are compared.
# So 'is CS worth it?' depends entirely on campus: at Davis you trade ~25pp of admit
# rate; at Irvine/Santa Cruz it barely moves.
hardest = result.iloc[0]
easiest = result.iloc[-1]
n_harder = int((result['cs_penalty_pp'] < 0).sum())
print(f"Hardest campus for CS applicants: {hardest['campus']} ({hardest['cs_penalty_pp']} pp penalty)")
print(f"Easiest campus for CS applicants: {easiest['campus']} ({easiest['cs_penalty_pp']} pp penalty)")
print(f"Campuses where CS is harder: {n_harder}/{len(result)}")
print(f"Mean penalty across {len(result)} campuses: {result['cs_penalty_pp'].mean():.1f} pp")

In [ ]:
# Build the chart used in the Streamlit dashboard
fig = px.bar(
    result.sort_values('cs_penalty_pp'),
    x='campus', y='cs_penalty_pp',
    color='cs_penalty_pp', color_continuous_scale='Reds',
    title='Fall 2025: Admit-Rate Penalty for Applying to Computer Science (CA Public HS Applicants)',
    labels={'cs_penalty_pp': 'CS penalty vs campus overall (pp)', 'campus': ''},
    text='cs_penalty_pp',
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.show()

In [ ]:
# RIGOR NOTE: We use campus-level discipline data (Fall 2025 only), not school-level.
# We do NOT average rates — penalty is computed per campus then described.
# CA-public filter applied via dashboard_data for campus-specific rates if needed;
# the discipline file is already systemwide by campus, which is the right grain for
# 'which campus punishes CS applicants the hardest'.
print('Methodology: campus-level, Fall 2025, penalty = CS_rate - AllDisciplines_rate.')
print('No school-level ethnicity summation. No fillna(0). Counts summed, then divided.')